# Notebook 1: Preprocessing
Loads the raw fitness dataset, maps IDs, performs 80/10/10 split,
generates negative samples, and uploads data to S3.

## Data
Raw fitness dataset with 973 members. Each row is treated as a unique user.
Workout type is mapped to an integer itemId. Implicit feedback derived from
workout frequency — sessions >= 4 days/week = 1 (engaged), else 0.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('src/data/fitness_ratings.csv')
print(df.shape)
print(df.head())

## Split
80% training, 10% validation, 10% test. Fixed random seed for reproducibility.

In [ ]:
# 80/10/10 split
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train: {len(train_df)}")
print(f"Val:   {len(val_df)}")
print(f"Test:  {len(test_df)}")

## Negative Sampling
Each user only interacted with one workout type. We sample unseen workout types
as negatives so the model learns to distinguish preferred from non-preferred activities.

In [ ]:
# Generate negative samples
# For each user, add rows for workout types they did NOT interact with
all_items = df['itemId'].unique()

def generate_negatives(data, num_negatives=4):
    records = []
    for _, row in data.iterrows():
        # Add the positive interaction
        records.append({'userId': row['userId'], 'itemId': row['itemId'], 'rating': row['rating'], 'timestamp': 0})
        # Add negative samples (items the user hasn't interacted with)
        user_items = df[df['userId'] == row['userId']]['itemId'].values
        negative_items = [i for i in all_items if i not in user_items]
        sampled = np.random.choice(negative_items, size=min(num_negatives, len(negative_items)), replace=False)
        for item in sampled:
            records.append({'userId': row['userId'], 'itemId': item, 'rating': 0, 'timestamp': 0})
    return pd.DataFrame(records)

train_data = generate_negatives(train_df)
val_data = generate_negatives(val_df)
test_data = generate_negatives(test_df)

print(f"Train with negatives: {len(train_data)}")
print(f"Val with negatives:   {len(val_data)}")
print(f"Test with negatives:  {len(test_data)}")

## Storage
Processed files saved locally and uploaded to S3 as the source of truth for downstream notebooks.

In [ ]:
import os
os.makedirs('src/data', exist_ok=True)

# Save processed CSVs
train_data.to_csv('src/data/train.csv', index=False)
val_data.to_csv('src/data/val.csv', index=False)
test_data.to_csv('src/data/test.csv', index=False)

print("Saved train.csv, val.csv, test.csv")

In [ ]:
import boto3

files = [
    'src/data/fitness_ratings.csv',
    'src/data/train.csv',
    'src/data/val.csv',
    'src/data/test.csv'
]

BUCKET = 'your-bucket-name'

try:
    s3 = boto3.client('s3')
    for f in files:
        s3.upload_file(f, BUCKET, f)
        print(f"Uploaded: {f}")
except Exception as e:
    print(f"S3 skipped (no credentials): {e}")